# Day 12 - AI Agents II: Multi-Step Agent Workflow

Build on the Day 11 LangChain + Gemini agent. This notebook runs a transparent workflow: **Research → Summarise → Save to File**. It records observable actions and tool activity, not private model reasoning.

## 1. Day 12 objectives

- Reuse Day 11's Gemini-backed LangChain agent and its tools.
- Perform a multi-step topic brief, structured summary, and file save.
- Record timestamped inputs, outputs, tool activity, decisions, and errors.
- Verify the final Markdown file and execution log.

## 2. Day 11 implementation analysis

The source notebook is `WEEK3/DAY11/agent/agent.ipynb`. Its implementation uses:

- **LLM:** `ChatGoogleGenerativeAI` with model `gemini-3.5-flash`.
- **Configuration:** Day 11 passes `api_key="key"` directly to the model constructor. Day 12 replaces that placeholder with the `GOOGLE_API_KEY` environment variable.
- **Agent:** `langchain.agents.create_agent(llm, tools)` and `agent.invoke({"messages": [...]})`.
- **Tools:** `get_word_count(text)` and `reverse_text(text)`, defined with `@tool` from `langchain_core.tools`.
- **Workflow:** the model can call a tool, receive its result, and produce a final answer. The notebook also demonstrates a general-knowledge prompt that does not need a tool.
- `GoogleGenerativeAIEmbeddings` is imported in Day 11 but not used in that notebook's agent workflow.

**Important limitation:** Day 11 does not configure web search, a browser, or a research API. This notebook reuses its Gemini model, agent setup, and tools. The research step asks Gemini for a topic brief from its model knowledge; it is **not live web research**, and answers should be independently checked when current facts or citations matter. The Day 11 text tools remain available to the agent but are not presented as search tools.

## 3. Multi-step agent architecture

1. **Research:** ask the Day 11 agent for a focused, clearly scoped topic brief.
2. **Summarise:** pass that brief to the same Gemini model for a structured summary.
3. **Save:** write the final Markdown summary to a topic-named file.

Each step logs its timestamp, action, input, and output. A LangChain callback logs tool start, result, and error events. A decision log records whether the model called a Day 11 tool; it does not expose chain-of-thought.

## 4. Required imports and configuration

Use the Python environment selected for this notebook. If a package is missing, install the same libraries used by Day 11 in a notebook cell with `%pip install langchain langchain-google-genai`. Set `GOOGLE_API_KEY` in your environment before a live model run. The code never embeds or logs the key. Without a key, the later sample cell runs a clearly labeled offline smoke test of the workflow and file-writing path.

In [1]:
import logging
import os
import re
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

OUTPUT_DIR = Path.cwd() / "day12_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / "day12_agent.log"

logger = logging.getLogger("day12_multi_step_agent")
logger.setLevel(logging.INFO)
if not logger.handlers:
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")
    file_handler.setFormatter(formatter)
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

def log_event(step: str, action: str, details: str) -> None:
    """Write a timestamped, observable workflow event to console and log file."""
    logger.info("step=%s | action=%s | %s", step, action, details)

@tool
def get_word_count(text: str) -> str:
    """Count the number of words in the given text."""
    return f"{len(text.split())} words"

@tool
def reverse_text(text: str) -> str:
    """Reverse the given text."""
    return text[::-1]

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
if GOOGLE_API_KEY:
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.5-flash",
        api_key=GOOGLE_API_KEY,
    )
    tools = [reverse_text, get_word_count]
    agent = create_agent(llm, tools)
    log_event("configuration", "ready", "Gemini and the Day 11 tools are configured.")
else:
    llm = None
    agent = None
    log_event(
        "configuration",
        "offline mode",
        "GOOGLE_API_KEY is not set; live model calls will require it.",
    )

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
2026-11-11 23:35:10,232 | INFO | step=configuration | action=ready | Gemini and the Day 11 tools are configured.


## 5. Research step

Keep the request focused and ask the model to distinguish established concepts from uncertainty. The callback reports actual Day 11 tool calls, if any; the agent is not forced to use irrelevant word-count or reversal tools.

In [2]:
class WorkflowTraceCallback(BaseCallbackHandler):
    """Log observable LangChain tool events without exposing private reasoning."""

    def on_tool_start(
        self,
        serialized: dict[str, Any],
        input_str: str,
        **kwargs: Any,
    ) -> None:
        tool_name = serialized.get("name", "unknown tool")
        log_event("research", "tool start", f"tool={tool_name}; input={input_str}")

    def on_tool_end(self, output: Any, **kwargs: Any) -> None:
        log_event("research", "tool output", str(output))

    def on_tool_error(self, error: BaseException, **kwargs: Any) -> None:
        log_event("research", "tool error", repr(error))

trace_callback = WorkflowTraceCallback()

def content_to_text(content: Any) -> str:
    """Convert common Gemini/LangChain text content shapes into plain text."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        text_parts = [
            item["text"]
            for item in content
            if isinstance(item, dict) and isinstance(item.get("text"), str)
        ]
        if text_parts:
            return "\n".join(text_parts)
    raise TypeError(f"Expected text content from the model, received {type(content).__name__}.")

def research_topic(topic: str) -> str:
    """Use the Day 11 Gemini agent to prepare a topic brief from model knowledge."""
    if agent is None:
        error = RuntimeError(
            "Live research requires GOOGLE_API_KEY. Set it in the environment and rerun configuration."
        )
        log_event("research", "error", repr(error))
        raise error

    prompt = (
        f"Prepare a concise research brief about: {topic}. "
        "Use your existing knowledge only; do not claim to browse or cite sources you did not access. "
        "Organize the brief into key ideas, practical examples, and uncertainties or facts to verify."
    )
    log_event("research", "input", f"topic={topic!r}; prompt={prompt}")
    try:
        response = agent.invoke(
            {"messages": [("user", prompt)]},
            config={"callbacks": [trace_callback]},
        )
        messages = response.get("messages", [])
        tool_calls = [
            call
            for message in messages
            for call in getattr(message, "tool_calls", [])
        ]
        if tool_calls:
            names = [call.get("name", "unknown tool") for call in tool_calls]
            log_event("research", "decision", f"agent called Day 11 tool(s): {names}")
        else:
            log_event(
                "research",
                "decision",
                "agent answered without calling a Day 11 utility; response uses model knowledge, not web search.",
            )
        if not messages:
            raise ValueError("The agent returned no messages.")
        research_brief = content_to_text(messages[-1].content)
        log_event("research", "output", research_brief)
        return research_brief
    except Exception as error:
        log_event("research", "error", repr(error))
        raise

## 6. Summarisation step

Pass only the collected brief into a second model call. The prompt requests a readable structure and explicitly asks the model not to add unsupported facts.

In [3]:
def summarise_research(topic: str, research_brief: str) -> str:
    """Turn the research brief into a structured Markdown summary."""
    if llm is None:
        error = RuntimeError(
            "Live summarisation requires GOOGLE_API_KEY. Set it in the environment and rerun configuration."
        )
        log_event("summarise", "error", repr(error))
        raise error

    prompt = (
        f"Create a clear, concise Markdown summary about {topic!r} using only the research brief below. "
        "Use headings: Overview, Key Points, Practical Takeaways, and What to Verify. "
        "Do not invent citations or introduce facts that are not supported by the brief.\n\n"
        f"Research brief:\n{research_brief}"
    )
    log_event("summarise", "input", f"topic={topic!r}; research_brief={research_brief}")
    try:
        response = llm.invoke(prompt)
        summary = content_to_text(response.content).strip()
        if not summary:
            raise ValueError("The model returned an empty summary.")
        log_event("summarise", "output", summary)
        return summary
    except Exception as error:
        log_event("summarise", "error", repr(error))
        raise

## 7. Save-to-file step

Write one UTF-8 Markdown file per topic beneath `day12_outputs`. The topic is sanitized before it becomes part of a filename. Files are created in the notebook's current working directory, which Jupyter normally sets to the notebook folder.

In [4]:
def save_summary(topic: str, summary: str, output_dir: Path = OUTPUT_DIR) -> Path:
    """Save a topic summary as a Markdown file and return its path."""
    if not summary.strip():
        error = ValueError("Summary must not be empty.")
        log_event("save", "error", repr(error))
        raise error
    safe_topic = re.sub(r"[^A-Za-z0-9._-]+", "-", topic.strip().lower()).strip("-._")
    if not safe_topic:
        safe_topic = "topic-summary"
    safe_topic = safe_topic[:80]
    output_path = output_dir / f"{safe_topic}.md"
    content = f"# Summary: {topic.strip()}\n\n{summary.strip()}\n"

    log_event("save", "input", f"path={output_path}; characters={len(content)}")
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        output_path.write_text(content, encoding="utf-8")
        log_event("save", "output", f"saved={output_path}; bytes={output_path.stat().st_size}")
        return output_path
    except Exception as error:
        log_event("save", "error", repr(error))
        raise

## 8. Complete multi-step agent workflow

The orchestrator runs each step in order, logs its inputs and outputs, and re-raises failures after recording them. Optional function arguments allow a deterministic offline smoke test without pretending it contacted Gemini.

In [5]:
def run_multistep_workflow(
    topic: str,
    output_dir: Path = OUTPUT_DIR,
    research_fn=None,
    summarise_fn=None,
) -> dict[str, Any]:
    """Run research, summarisation, and file saving with timestamped traces."""
    topic = topic.strip()
    if not topic:
        error = ValueError("Topic must not be empty.")
        log_event("workflow", "error", repr(error))
        raise error

    research_fn = research_fn or research_topic
    summarise_fn = summarise_fn or summarise_research
    log_event("workflow", "start", f"topic={topic!r}")
    try:
        research_brief = research_fn(topic)
        summary = summarise_fn(topic, research_brief)
        output_path = save_summary(topic, summary, output_dir)
        result = {
            "topic": topic,
            "research_brief": research_brief,
            "summary": summary,
            "output_path": output_path,
        }
        log_event("workflow", "complete", f"topic={topic!r}; output={output_path}")
        return result
    except Exception as error:
        log_event("workflow", "error", f"topic={topic!r}; error={error!r}")
        raise

## 9. Execution logs and observable traces

Every log line has a timestamp and step name. The trace includes the topic and prompts, returned research brief and summary, file path, tool inputs/outputs/errors, model tool-use decision, and workflow errors. These are observable execution events, not private internal reasoning. Logs are written to `day12_outputs/day12_agent.log` and the notebook output.

## 10. Test with sample topics

With `GOOGLE_API_KEY` configured this runs the live Gemini agent and summariser. Without a key, it clearly switches to deterministic sample functions so the entire orchestration and save path can still be tested offline. The offline result is not live research.

In [6]:
sample_topics = [
    "How AI agents use tools",
    "Retrieval-augmented generation",
]
topic = sample_topics[0]

if agent is not None and llm is not None:
    workflow_result = run_multistep_workflow(topic)
else:
    print("OFFLINE SMOKE TEST: no live Gemini request will be made.")

    def sample_research(sample_topic: str) -> str:
        brief = (
            f"Sample brief for {sample_topic}: an AI agent can select a suitable tool, "
            "observe its result, and use that result to complete a task. "
            "This deterministic text is test data, not a live research result."
        )
        log_event("research", "offline sample output", brief)
        return brief

    def sample_summarise(sample_topic: str, brief: str) -> str:
        summary = (
            "## Overview\n"
            f"This offline sample describes {sample_topic}.\n\n"
            "## Key Points\n"
            "- Agents can choose tools to perform concrete actions.\n"
            "- The agent can use a tool result in its final response.\n\n"
            "## Practical Takeaways\n"
            "Use a tool when a task needs a precise operation or external action.\n\n"
            "## What to Verify\n"
            "Replace this offline sample with a live model run for a real topic brief."
        )
        log_event("summarise", "offline sample input", brief)
        log_event("summarise", "offline sample output", summary)
        return summary

    workflow_result = run_multistep_workflow(
        topic,
        research_fn=sample_research,
        summarise_fn=sample_summarise,
    )

print("Workflow output:", workflow_result["output_path"])
print(workflow_result["summary"])

2026-11-11 23:35:10,316 | INFO | step=workflow | action=start | topic='How AI agents use tools'
2026-11-11 23:35:10,317 | INFO | step=research | action=input | topic='How AI agents use tools'; prompt=Prepare a concise research brief about: How AI agents use tools. Use your existing knowledge only; do not claim to browse or cite sources you did not access. Organize the brief into key ideas, practical examples, and uncertainties or facts to verify.
2026-11-11 23:35:18,569 | INFO | step=research | action=decision | agent answered without calling a Day 11 utility; response uses model knowledge, not web search.
2026-11-11 23:35:18,570 | INFO | step=research | action=output | # Research Brief: How AI Agents Use Tools

---

## 1. Key Ideas

* **Augmenting Model Capabilities:** Large Language Models (LLMs) are fundamentally text generators with finite training cutoffs and limited math/execution capabilities. Tool use bridges this gap by allowing agents to interact with external software, APIs,

Workflow output: e:\Ivoyant\WEEK3\DAY12\agent\day12_outputs\how-ai-agents-use-tools.md
# How AI Agents Use Tools

## Overview
Large Language Models (LLMs) are text generators constrained by finite training cutoffs and limited math or execution capabilities. Tool use bridges these limitations, enabling agents to interact directly with external software, APIs, databases, and execution environments to complete complex tasks.

## Key Points
* **Function Calling & Schema Binding:** Agents use predefined structured tool definitions (often JSON schemas) to parse requests, determine necessary tool calls, and output structured parameters containing the tool name and arguments.
* **The Agentic Loop:** Tool use follows a structured cycle:
  1. *Thought / Plan:* The agent evaluates the objective and chooses a tool.
  2. *Action:* The agent generates a formatted tool execution request.
  3. *Observation:* The execution environment runs the tool and returns the results to the context window.
  4. *R

## 11. Final output verification

Check that the summary exists, is non-empty, and includes the topic heading. The execution log should also exist.

In [7]:
output_path = workflow_result["output_path"]
assert output_path.is_file(), f"Summary file was not created: {output_path}"
saved_text = output_path.read_text(encoding="utf-8")
assert saved_text.strip(), "Summary file is empty."
assert saved_text.startswith(f"# Summary: {workflow_result['topic']}")
assert LOG_PATH.is_file(), f"Execution log was not created: {LOG_PATH}"

print(f"Verified summary: {output_path.resolve()}")
print(f"Summary size: {output_path.stat().st_size} bytes")
print(f"Verified execution log: {LOG_PATH.resolve()}")
print("Saved preview:\n")
print(saved_text[:1000])

Verified summary: E:\Ivoyant\WEEK3\DAY12\agent\day12_outputs\how-ai-agents-use-tools.md
Summary size: 2908 bytes
Verified execution log: E:\Ivoyant\WEEK3\DAY12\agent\day12_outputs\day12_agent.log
Saved preview:

# Summary: How AI agents use tools

# How AI Agents Use Tools

## Overview
Large Language Models (LLMs) are text generators constrained by finite training cutoffs and limited math or execution capabilities. Tool use bridges these limitations, enabling agents to interact directly with external software, APIs, databases, and execution environments to complete complex tasks.

## Key Points
* **Function Calling & Schema Binding:** Agents use predefined structured tool definitions (often JSON schemas) to parse requests, determine necessary tool calls, and output structured parameters containing the tool name and arguments.
* **The Agentic Loop:** Tool use follows a structured cycle:
  1. *Thought / Plan:* The agent evaluates the objective and chooses a tool.
  2. *Action:* The agent

## 12. Key learnings and conclusion

- A multi-step workflow separates information gathering, summarisation, and persistence into observable stages.
- Day 11's Gemini model and LangChain tools can be reused, but the original notebook has no web-search capability; model knowledge is not live research.
- Timestamped logs and tool callbacks make actions, inputs, outputs, decisions, and failures easier to monitor.
- Environment variables keep API keys out of notebook source. Errors are logged and re-raised rather than hidden.
- The final Markdown file and execution log are saved under `day12_outputs` in the notebook's working directory.